# BrainFormer v13-v2 — The Deep Brain (Kaggle GPU T4 x2)

A **deep, no-backprop** brain-style language model vs a real pretrained LLM (Pythia-410M) on WikiText-103.

- Stack of locally-trained cerebellar layers (delta-rule deep supervision, learned mixture)
- Hyperdimensional long-context binding (ctx 6 -> 32, no fan-in dilution)
- Continuous competitive granule learning
- Fused activation, two-level approx top-k, flat scatter-add, async prefetch

**Settings to set before running:** Accelerator = `GPU T4 x2`, Internet = `On`.

Order: smoke (sanity) -> build-data (tokenize once) -> full dual-GPU run.

## 1. Get the code (clone or refresh)

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
# One-time: Kaggle -> Add-ons -> Secrets -> add a secret named GITHUB_TOKEN.
tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
url = f'https://{tok}@github.com/PlangoDev/plango-labs.git'
try:
    if not os.path.isdir('plango-labs'):
        subprocess.run(['git', 'clone', '--depth', '1', url],
                       check=True, capture_output=True)  # token not echoed
    else:
        subprocess.run(['git', '-C', 'plango-labs', 'pull', '--ff-only'],
                       check=True, capture_output=True)
except subprocess.CalledProcessError:
    raise RuntimeError('git clone/pull failed - check the GITHUB_TOKEN secret') from None
finally:
    del tok, url  # keep the token out of notebook state
%cd plango-labs/test_013_v2
!ls -1 bf

## 2. Environment check (two T4s, libraries)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPUs', torch.cuda.device_count())
# transformers + datasets ship on Kaggle; uncomment to pin/upgrade if needed.
# !pip -q install -U transformers datasets

## 3. Smoke test (CPU-OK, ~1-2 min) — exercises the full multi-layer path on tiny-gpt2

If this prints a SCOREBOARD with finite perplexities, the pipeline is sound.

In [ ]:
!python run.py --smoke

## 4. Build the token memmap ONCE (streams Wikipedia; needs Internet ON)

Cached under `/kaggle/working`, so it survives notebook restarts and is reused by the run below.
Default budget is 500M tokens; start smaller to iterate fast, e.g. `--train-tokens 50000000`.

In [ ]:
!python run.py --build-data --train-tokens 50000000

## 5. Train + score (auto dual-GPU)

Match `--train-tokens` to the memmap you built. The scoreboard prints per-layer
perplexity and the learned mixture weights, so you can see which depths do the work.

VRAM knobs if you OOM: lower `--n-gran` (granules per layer) or `--n-layers`.
Context curve: try `--ctx 6` / `--ctx 32` / `--ctx 64`.

In [ ]:
!python run.py --train-tokens 50000000

## 6. Ablations (optional)

Each isolates one lever. Run after a baseline so the comparison is apples-to-apples.

In [ ]:
# Depth gain: collapse to a single layer with the new front-end
!python run.py --train-tokens 50000000 --n-layers 1
# Continuous-learning gain: freeze granules after setup (v13 behavior)
# !python run.py --train-tokens 50000000 --no-refine
# Long-context curve
# !python run.py --train-tokens 50000000 --ctx 64

## 7. Full 500M-token run (the headline number)

Rebuild the memmap at the full budget first (one-time), then train.

In [ ]:
# !python run.py --build-data --train-tokens 500000000
# !python run.py --train-tokens 500000000